# Passing structured invoice data with `context=` in Deep Agents

This notebook demonstrates how to pass structured business data (invoice IDs) into a Deep Agent **without** embedding them in the user message.

**Pattern:** define `context_schema=InvoiceContext` at agent creation, pass `context={"invoice_ids": [...]}` at invoke time. Tools read scoped IDs from `runtime.context`. A `@dynamic_prompt` middleware reads the same `context=` at invoke time and injects scoped IDs into the system prompt so the model knows which IDs to pass to tools.

Runtime context is immutable for the run and is **not** automatically added to the model prompt — use middleware (or a discovery tool) when the model needs to see scoped IDs.

In [1]:
from pprint import pprint

from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model

from invoice_context_demo.demo_support import (
    InvoiceContext,
    build_analyze_invoice_tool,
    build_fake_invoices,
    invoice_scope_prompt,
    pretty_print_messages,
    require_openai_api_key,
)

require_openai_api_key()

MODEL = init_chat_model("openai:gpt-4.1-mini", temperature=0)

INVOICE_REGISTRY = build_fake_invoices()
INVOICE_IDS = list(INVOICE_REGISTRY)
USER_MESSAGE = "Please analyze the attached invoices."

print("Fake invoices in registry:")
for invoice_id, record in INVOICE_REGISTRY.items():
    print(f"  {invoice_id}: {record.vendor} (${record.amount_usd:,.2f}, {record.status})")

Fake invoices in registry:
  INV-001: Acme Office Supplies ($1,248.50, pending)
  INV-002: CloudScale Hosting ($3,890.00, approved)
  INV-003: Metro Catering Co. ($612.75, flagged)


## Run the agent

The user message stays generic. Invoice IDs are passed in the structured `context=` payload. `invoice_scope_prompt` middleware and the `analyze_invoice` tool both read from the same runtime context.

In [2]:
analyze_invoice = build_analyze_invoice_tool(INVOICE_REGISTRY)

agent = create_deep_agent(
    model=MODEL,
    tools=[analyze_invoice],
    context_schema=InvoiceContext,
    middleware=[invoice_scope_prompt],
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": USER_MESSAGE}]},
    config={"configurable": {"thread_id": "invoice-demo-context"}},
    context={"invoice_ids": INVOICE_IDS},
)

print("User message (no invoice IDs embedded):")
print(f"  {USER_MESSAGE}\n")
print("Structured payload passed via context=")
pprint({"invoice_ids": INVOICE_IDS})
print("\nAgent run:")
pretty_print_messages(result)

User message (no invoice IDs embedded):
  Please analyze the attached invoices.

Structured payload passed via context=
{'invoice_ids': ['INV-001', 'INV-002', 'INV-003']}

Agent run:
AIMessage → tool_call analyze_invoice({"invoice_id": "INV-001"})
AIMessage → tool_call analyze_invoice({"invoice_id": "INV-002"})
AIMessage → tool_call analyze_invoice({"invoice_id": "INV-003"})
ToolMessage: INV-001 | vendor=Acme Office Supplies | amount=$1,248.50 | status=pending | items=[Standing desks x4, Monitor arms x4]
ToolMessage: INV-002 | vendor=CloudScale Hosting | amount=$3,890.00 | status=approved | items=[GPU instances (March), Object storage]
ToolMessage: INV-003 | vendor=Metro Catering Co. | amount=$612.75 | status=flagged | items=[Team lunch, Late delivery fee]
AIMessage: I have analyzed the attached invoices:

1. Invoice INV-001 is from Acme Office Supplies for $1,248.50. It is currently pending and includes items such as 4 standing desks and 4 monitor arms.
2. Invoice INV-002 is from Clou

## Reference

Docs: [Context engineering in Deep Agents](https://docs.langchain.com/oss/python/deepagents/context-engineering)